# Churn Prediction — Exploratory Analysis

Exploratory walkthrough covering data profiling, preprocessing, quick model training, and SHAP explainability.  
For the full production run (Optuna tuning + MLflow + gate check), use `run_churn.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.churn import (
    AUC_ROC_GATE,
    PRECISION_TOP20_GATE,
    load_and_preprocess,
    precision_at_top_k,
)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')

## 1. Data Exploration

In [ ]:
df = pd.read_csv(ROOT / 'data/raw/online_retail_customer_churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
df.describe()

### Target Distribution

In [ ]:
counts = df['Target_Churn'].value_counts()
print(counts)
print(f'\nChurn rate: {counts[True] / len(df) * 100:.1f}%')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['No Churn', 'Churn'], [counts[False], counts[True]], color=['steelblue', 'salmon'])
ax.set_title('Target Distribution')
ax.set_ylabel('Count')
for i, v in enumerate([counts[False], counts[True]]):
    ax.text(i, v + 50, str(v), ha='center')
plt.tight_layout()
plt.show()

### Feature Distributions by Churn Status

In [ ]:
num_cols = [
    'Age', 'Annual_Income', 'Total_Spend', 'Years_as_Customer',
    'Num_of_Purchases', 'Average_Transaction_Amount', 'Num_of_Returns',
    'Num_of_Support_Contacts', 'Satisfaction_Score', 'Last_Purchase_Days_Ago',
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, col in zip(axes.flat, num_cols):
    for churn_val, color, label in [(False, 'steelblue', 'No Churn'), (True, 'salmon', 'Churn')]:
        ax.hist(df.loc[df['Target_Churn'] == churn_val, col], bins=25,
                alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle('Feature Distributions by Churn Status', fontsize=13)
plt.tight_layout()
plt.show()

### Correlation Heatmap

In [ ]:
corr_df = df[num_cols].copy()
corr_df['Target_Churn'] = df['Target_Churn'].astype(int)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
X, y, cids = load_and_preprocess(ROOT / 'data/raw/online_retail_customer_churn.csv')

print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')
print(f'Churn rate: {y.mean():.3f}')
print(f'\nEngineered features added:')
for f in ('return_rate', 'spend_per_year', 'support_per_purchase'):
    print(f'  {f}: {X[f].describe()["mean"]:.4f} mean')

In [ ]:
X.head()

## 3. Train / Test Split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y.values, test_size=0.20, stratify=y.values, random_state=42
)
print(f'Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'Train churn rate: {y_tr.mean():.3f}')
print(f'Test  churn rate: {y_te.mean():.3f}')

## 4. Quick Model Training (Default Params)

Single XGBoost run with fixed params for exploration. Full Optuna tuning runs via `python run_churn.py`.

In [ ]:
spw   = float((y_tr == 0).sum() / (y_tr == 1).sum())
model = XGBClassifier(
    max_depth=5, learning_rate=0.1, n_estimators=200,
    scale_pos_weight=spw, tree_method='hist',
    random_state=42, n_jobs=-1, verbosity=0,
)
model.fit(X_tr, y_tr)
print('Training complete.')

In [ ]:
y_prob   = model.predict_proba(X_te)[:, 1]
auc_roc  = roc_auc_score(y_te, y_prob)
p_top20  = precision_at_top_k(y_te, y_prob, k=0.20)

status_auc  = '✓' if auc_roc  >= AUC_ROC_GATE         else '✗ BELOW GATE'
status_p20  = '✓' if p_top20  >= PRECISION_TOP20_GATE  else '✗ BELOW GATE'

print(f'AUC-ROC:          {auc_roc:.4f}   (gate ≥ {AUC_ROC_GATE})  {status_auc}')
print(f'Precision@top20:  {p_top20:.4f}   (gate ≥ {PRECISION_TOP20_GATE})  {status_p20}')

## 5. SHAP Explainability

In [ ]:
X_te_df   = pd.DataFrame(X_te, columns=X.columns.tolist())
explainer = shap.TreeExplainer(model)
sv        = explainer.shap_values(X_te_df)

shap.summary_plot(sv, X_te_df, feature_names=X.columns.tolist())

In [ ]:
shap.summary_plot(sv, X_te_df, feature_names=X.columns.tolist(), plot_type='bar')

### Waterfall — Single Prediction Breakdown

In [ ]:
shap_exp  = explainer(X_te_df)
churn_idx = int(np.where(y_te == 1)[0][0])

print(f'Predicted churn probability: {y_prob[churn_idx]:.4f}')
shap.plots.waterfall(shap_exp[churn_idx])

In [ ]:
no_churn_idx = int(np.where(y_te == 0)[0][0])

print(f'Predicted churn probability: {y_prob[no_churn_idx]:.4f}')
shap.plots.waterfall(shap_exp[no_churn_idx])

### Dependence Plot — Top Feature

In [ ]:
top_idx  = int(np.abs(sv).mean(axis=0).argmax())
top_feat = X.columns[top_idx]
print(f'Top feature by mean |SHAP|: {top_feat}')

shap.dependence_plot(top_idx, sv, X_te_df, feature_names=X.columns.tolist())

## 6. Production Run

Run the full pipeline with Optuna tuning (20 trials), 5-fold CV, MLflow tracking, and acceptance gate check:

```bash
python run_churn.py --n-trials 20
```

Acceptance gates:
- AUC-ROC ≥ 0.88
- Precision@top20 ≥ 0.75

Exits with code 1 and prints a warning to stderr if either gate fails. All artifacts (model, plots, predictions) are logged to MLflow regardless.